# Amazon ML Challenge 2026: Business Entity Resolution

Our end-to-end pipeline, from the raw TSVs to `matching_results.tsv` + `candidate_pairs.tsv`.

```
 raw TSVs ─► normalize (learned translit + abbreviation maps) ─► fine-tune multilingual bi-encoder (contrastive)
                                                                         │
     ┌───────────────────────────────────────────────────────────────────┘
     ▼
 BLOCKING per country: dense top-8 (FAISS IVF) ∪ TF-IDF name-char / name-word / address top-4
     ▼
 STAGE 1  XGBoost(GPU) on cheap scores + ranks + margins  ─► keep top-3 candidates per S2/S3 record  ═► candidate_pairs.tsv
     ▼
 STAGE 2  XGBoost(GPU) on 40+ fuzzy/number/context features  (+ optional cross-encoder on the ambiguous band)
     ▼
 ASSIGNMENT (each S2/S3 record → at most one S1)  ─►  per-S1 expected-F0.5 subset selection, tuned on a held-out world ═► matching_results.tsv
```

### Facts we measured on the data (they drive the design)
| Fact | Consequence |
|---|---|
| Train: 2.21M S1, 5.03M S2, 5.29M S3. Test: 1.73M S1, 4.89M S2, 5.08M S3 | ~12M records per split. Everything is vectorised, runs per country, and is checkpointed |
| **Every S2/S3 record matches at most one S1** (0 duplicates across 7.64M matched ids) | We score (other → S1) and run an **assignment**, keeping at most one S1 per other. Rank/margin features among an other's candidates are very strong |
| **Matches never cross countries** (0 of 693k checked pairs) | Blocking runs inside each country label. Country is never a model feature, so France works |
| 5.6% of S1 are singletons; the per-S1 match count peaks at 3–4 (max 9+) | Per-S1 subset selection has to be able to output "empty" |
| 26% of train S2/S3 records match nothing. Test has 5.76 others per S1 vs 4.68 in train | Test has **more distractors**. Our validation world is topped up to the test ratio so its precision is honest |
| S1 is always Latin script. S2/S3 Indian names/addresses are often in Devanagari/Tamil/Kannada/Bengali… | We learn native→Latin token maps from train pairs, use a consonant skeleton, and a multilingual encoder |
| Noise: typos, `Énterprises`, leet (`Chemica1s`), domains (`maurewilliamscolombier.com`), legal-suffix shuffles, `40ND STREET`, `null` components, empty addresses (~3%), trade names with a totally different name but the same address | Separate name / compact-name / address views, ordinal and number normalisation, and an address-only blocking view |

### How to run on Kaggle
1. **Dataset:** zip the `student_resource/dataset` folder (train/ + test/) plus `utils/validate_submission.py`, then upload it as a private Kaggle Dataset and *Add Input* it to this notebook. Paths are auto-detected.
2. **Settings:** Accelerator **GPU T4 x2** (P100 does not work with current torch), **Internet ON** (pip + HuggingFace weights), Persistence off.
3. **Dry run first:** set `SMOKE = True` in the config cell and *Run All* (~10–15 min on a tiny sample). This catches environment problems without burning 4 hours.
4. **Full run:** `SMOKE = False`, then **Save Version → Save & Run All (Commit)**. A headless run survives browser disconnects. Expect roughly 3–5 h. Outputs land in `/kaggle/working/output/`.
5. Download `output/*.tsv`, check that the validator cell printed **PASS**, and upload `matching_results.tsv` to the portal.

## 0. Setup
`rapidfuzz` (fast vectorised fuzzy matching), `sparse_dot_topn` (multi-threaded sparse top-k) and `unidecode` are not preinstalled on Kaggle. torch, transformers, xgboost and scikit-learn are.

In [1]:
!pip install -q "rapidfuzz>=3.6" "sparse_dot_topn>=1.1" unidecode faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.6/309.6 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 73.6 MB/s eta 0:00:00


In [2]:
import os, re, gc, csv, glob, json, math, time, shutil, unicodedata
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # avoids GPU fragmentation OOMs
from collections import Counter
from multiprocessing import Pool
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import xgboost as xgb
import faiss
from rapidfuzz import fuzz, process
from rapidfuzz.distance import JaroWinkler
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn
from unidecode import unidecode

SMOKE = False            # True = quick end-to-end dry run on ~2% of the data
VERSION = 'final-1 (v5): sibling worlds + CE wide band + 3-seed stage-2 + set ranker, TRAIN_FRAC 0.25'

CFG = dict(
    SEED=42,
    TRAIN_FRAC=0.25,     # share of train S1 used to FIT the models (world A)
    VALID_FRAC=0.10,     # disjoint share held out for tuning + honest scoring (world B)
    OTHER_RATIO=5.76,    # |S2∪S3| / |S1| in test; both worlds are topped up to this distractor density
    # bi-encoder (MIT licence, 118M params)
    EMB_MODEL='intfloat/multilingual-e5-small', EMB_MAXLEN=64, EMB_BS=256, EMB_LR=5e-5,
    EMB_EPOCHS=1, EMB_TEMP=0.05, EMB_FT_PAIRS=700_000,
    # blocking
    K_DENSE=8, K_SPARSE=4, SPARSE_MIN=0.1,
    NPROBE_FRAC=0.02, IVF_MIN=20_000,    # FAISS IVF probes 2% of clusters in every world, so B's recall is representative of test
    # stage-1 pruning -> this is the candidate set written to candidate_pairs.tsv
    KEEP_PER_OTHER=3, STAGE1_MIN_P=0.005, S1_TRAIN_ROWS=24_000_000,
    # optional cross-encoder on the ambiguous band (MIT licence, 278M params)
    # v4: on. It reads both records jointly, which suits siblings (one word / one number differs).
    # CE_DEADLINE_MIN: skip cross-encoder scoring for any country that starts after this many minutes
    RUN_CE=True, CE_DEADLINE_MIN=560, N_STAGE2=3,
    RUN_SET=True, SET_MAXLEN=32, SET_DIM=128, SET_LAYERS=3, SET_EPOCHS=5, SET_BS=512, SET_LR=1e-3, CE_MODEL='xlm-roberta-base', CE_TRAIN_PAIRS=250_000, CE_MAXLEN=96,
    CE_BS=64, CE_LR=2e-5, CE_BAND=(0.02, 0.995),
    SMOKE_FRAC=0.02,
)
CFG.update(json.loads(os.environ.get('BER_CFG', '{}')))   # optional overrides without editing the notebook
if SMOKE:
    CFG.update(TRAIN_FRAC=0.3 * CFG['SMOKE_FRAC'], VALID_FRAC=0.15 * CFG['SMOKE_FRAC'],
               EMB_FT_PAIRS=20_000, CE_TRAIN_PAIRS=20_000)

def _find(name):
    roots = [os.environ['BER_DATA']] if 'BER_DATA' in os.environ else ['/kaggle/input', '.', '..']
    for r in roots:
        hits = sorted(glob.glob(f'{r}/**/{name}', recursive=True))
        if hits:
            return Path(hits[0])
    raise FileNotFoundError(f'{name} not found - did you add the dataset as input?')

TRAIN_DIR = _find('train_source1.tsv').parent
TEST_DIR = _find('test_source1.tsv').parent
ON_KAGGLE = Path('/kaggle/working').exists()
WORK = Path('/kaggle/working') if ON_KAGGLE else Path(os.environ.get('BER_WORK', 'ber_work'))
TMP = Path('/tmp/ber') if ON_KAGGLE else WORK / 'tmp'      # big intermediates (not saved as Kaggle output)
OUT = WORK / 'output'
for p in (WORK, TMP, OUT):
    p.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
NGPU = torch.cuda.device_count()
NCPU = os.cpu_count()
faiss.omp_set_num_threads(NCPU)
XGB_DEV = 'cuda' if DEVICE == 'cuda' else 'cpu'
RNG = np.random.default_rng(CFG['SEED'])
torch.manual_seed(CFG['SEED'])

T0 = time.time()
def log(*a):
    print(f'[{(time.time() - T0) / 60:6.1f} min]', *a, flush=True)

log(VERSION)
log('train:', TRAIN_DIR, '| test:', TEST_DIR, '| device:', DEVICE, 'x', NGPU, '| cpus:', NCPU)

[   0.0 min] final-1 (v5): sibling worlds + CE wide band + 3-seed stage-2 + set ranker, TRAIN_FRAC 0.25
[   0.0 min] train: /kaggle/input/datasets/saumya170408/amazon-ml-challenge-26/student_resource/dataset/train | test: /kaggle/input/datasets/saumya170408/amazon-ml-challenge-26/student_resource/dataset/test | device: cuda x 2 | cpus: 4


## 1. Load and look at the data
Everything is read as strings with `quoting=QUOTE_NONE`, because names contain quotes and `null` is a literal token and must not become NaN. Ids become int64 keys (`S2-123` → `2·10¹⁰ + 123`), which keeps joins over 10M+ rows cheap.

In [3]:
def read_tsv(p):
    return pd.read_csv(p, sep='\t', dtype=str, keep_default_na=False, quoting=csv.QUOTE_NONE)

def ikey(ids):
    ids = pd.Series(ids, dtype=str)
    return (ids.str[1].astype(np.int64) * 10**10 + ids.str[3:].astype(np.int64)).to_numpy()

def skey(keys):
    keys = np.asarray(keys, dtype=np.int64)
    return 'S' + pd.Series(keys // 10**10).astype(str) + '-' + pd.Series(keys % 10**10).astype(str)

def load_split(split, d):
    parts = []
    for i in (1, 2, 3):
        df = read_tsv(d / f'{split}_source{i}.tsv')
        df['src'] = np.int8(i)
        parts.append(df)
    df = pd.concat(parts, ignore_index=True)
    df['key'] = ikey(df.entity_id)
    return df

train = load_split('train', TRAIN_DIR)
gt = read_tsv(TRAIN_DIR / 'train_ground_truth.tsv')
ex = gt.assign(o=gt.matched_entity_ids.str.split(',')).explode('o')
ex = ex[ex.o.notna() & (ex.o != '')]
gt_pairs = pd.DataFrame({'s1': ikey(ex.source1_entity_id), 'o': ikey(ex.o)})
del ex
log('train rows by source:', train.src.value_counts().sort_index().to_dict())
log('countries:', train.groupby(['src', 'country']).size().unstack(0).to_dict())
assert gt_pairs.o.is_unique, 'an S2/S3 record matched two S1s - the assignment step assumes this never happens'
n_match = gt.matched_entity_ids.map(lambda s: 0 if s == '' else s.count(',') + 1)
log('matches per S1:', n_match.value_counts().sort_index().to_dict())
log(f'singleton S1: {(n_match == 0).mean():.2%} | unmatched others: {1 - len(gt_pairs) / (train.src != 1).sum():.2%}')

[   1.2 min] train rows by source: {1: 2206821, 2: 5034616, 3: 5285603}
[   1.2 min] countries: {1: {'India': 883188, 'US': 1323633}, 2: {'India': 2017799, 'US': 3016817}, 3: {'India': 2115547, 'US': 3170056}}
[   1.3 min] matches per S1: {0: 123247, 1: 119157, 2: 375212, 3: 530841, 4: 484115, 5: 321957, 6: 164868, 7: 63968, 8: 18680, 9: 4205, 10: 534, 11: 37}
[   1.3 min] singleton S1: 5.58% | unmatched others: 25.99%


In [4]:
# A few ground-truth groups, to see the noise with our own eyes
_show = gt_pairs.drop_duplicates('s1').head(4).s1
_idx = train.set_index('key')
for s in _show:
    print(_idx.loc[s, ['business_name', 'business_address']].tolist())
    for o in gt_pairs[gt_pairs.s1 == s].o:
        print('    ', _idx.loc[o, ['business_name', 'business_address']].tolist())
del _idx

['Maure Williams Colombier Inc', '85 Wayne Avenue, Ticonderoga, NY']
     ['Maure Wilblims Colombier Inc', '']
     ['Maure Williams Colombier', '']
     ['Dréxkor', '85 Wanye Avenue, Ticonderoga Townshiip, New York']
     ['maurewilliamscolombier.com', 'Wayne Ave, Ticonderoga Townshiip, New York']
     ['Maure Williams Inc Center', '']
['Raj Investments LLP', '6(29), C.I.T. Colony, 2Nd Main Road Mylapore, Chennai, Tamil Nadu']
     ['ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி', '6(29), C.I.T. COLONY, 2ND MAIN ROAD MYLAPORE, CHENNAI, Tamil Nadu']
     ['Raj Investments LLP', '6(29), C.I.T. COLONY, 2ND MAIN ROAD MYLAPORE, CHENNAI, Tamil Nadu']
     ['Raj Investments எல்எல்பி', '6(29), C.i.t. Colony, 2Nd Main Road Mylapore, Chennai, TN']
     ['ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி', '6(29), C.i.t. Colony, 2Nd Main Road Mylapore, Chennai, தமிழ்நாடு']
['Dahlia Power Reliable Scientific LLC', '630 45th Terrace, Kansas City, MO']
     ['Dahlia Power Reliable', 'KANSAS CITY, MO, 630 45ND TERRACE, null']
    

## 2. Two disjoint "worlds" carved out of train
We can't score on test, so we need a validation set that behaves like test:

* S1 entities are split into **A (fit, 30%)** and **B (held out, 10%)** by hashing their **first name token** (per country). Each world gets its S1s and *all* their matched records.
* **Unmatched records follow the S1 they imitate.** The unmatched pool is mostly deliberate *siblings*: a copy of a real S1 with the same first name token, an extra descriptor word ("Group", "Industries", "Holdings") and a nearby house number. Each goes to the world holding its name group; Indic-script first tokens are first mapped to their Latin S1 token, learned from train pairs. The first version split these records at random, so only ~10–20% of siblings sat next to their S1. Validation said 0.987 while the leaderboard said 0.926.
* Test has 5.76 others per S1, train only 4.68, so test has more distractors. We **top each world up to 5.76** with records whose true S1 is in neither world. For that world they are exactly like an unmatched test record: the right answer does not exist.
* The remaining ~70% of train is left unused, which keeps the full run inside Kaggle's 12 h. Raise `TRAIN_FRAC` if you have time to spare.

In [5]:
def h01(values):
    # deterministic hash in [0, 1)
    return pd.util.hash_pandas_object(pd.Series(values), index=False).to_numpy() / 2.0**64

_NAT = re.compile(r'[\u0900-\u0DFF]')
_KEY_STOP = set("""mr mrs ms dr smt shri m s the llc inc ltd pvt co corp sas sarl sa sci eurl sasu sa and""".split())

def name_first_token(names):
    # first meaningful name token; siblings ("Radha Technologies Group") share it with the S1 they copy
    out = []
    for n in names:
        n = ''.join(c for c in unicodedata.normalize('NFKD', n) if not unicodedata.combining(c)).lower()
        toks = re.findall(r'[\w\u0900-\u0DFF]+', n)          # the Indic range keeps vowel signs inside words
        toks = [t for t in toks if len(t) > 1 and t not in _KEY_STOP and not t.isdigit()]
        out.append(toks[0] if toks else '')
    return np.array(out, dtype=object)

def make_worlds(df, pairs):
    """Worlds are cut along S1 *name groups*, and every unmatched record follows the S1 group it resembles.

    Unmatched records are mostly deliberate near-copies ("siblings") of a real S1: same first name
    token, extra descriptor word, nearby house number. Splitting them at random (the first version)
    left only ~10-20% of them next to their S1, so validation overestimated precision (0.987 vs 0.926 LB).
    """
    s1 = df[df.src == 1]
    oth = df[df.src != 1]
    tf, vf = CFG['TRAIN_FRAC'], CFG['VALID_FRAC']
    k1 = s1.country.values + '|' + name_first_token(s1.business_name.values)
    ko = oth.country.values + '|' + name_first_token(oth.business_name.values)
    # native-script first tokens -> their Latin S1 token, learned from train pairs
    s1k = pd.Series(k1, index=s1.key.values)
    m = pd.DataFrame({'ko': ko, 'o': oth.key.values}).merge(pairs, on='o')
    m = m[[bool(_NAT.search(x)) for x in m.ko]]
    if len(m):
        m['k1'] = m.s1.map(s1k)
        nat = m.groupby(['ko', 'k1']).size().rename('n').reset_index().sort_values('n').drop_duplicates('ko', keep='last')
        ko = pd.Series(ko).map(dict(zip(nat.ko, nat.k1))).fillna(pd.Series(ko)).to_numpy()
    u = h01(k1)
    s1_world = pd.Series(np.select([u < tf, u < tf + vf], ['A', 'B'], '-'), index=s1.key.values)
    key_world = pd.Series(s1_world.values, index=k1)
    key_world = key_world[~key_world.index.duplicated()]
    owner = pd.Series(pairs.s1.values, index=pairs.o.values)
    ow = pd.Series(oth.key.map(owner).map(s1_world).to_numpy(), index=oth.index)
    unmatched = ow.isna().to_numpy()
    via_key = pd.Series(ko).map(key_world).to_numpy()
    has_key = unmatched & pd.notna(via_key)
    ow[has_key] = via_key[has_key]
    r = RNG.random(len(oth))
    rest = unmatched & ~has_key
    ow[rest & (r < tf)] = 'A'
    ow[rest & (r >= tf) & (r < tf + vf)] = 'B'
    ow[rest & (r >= tf + vf)] = '-'
    log(f'unmatched records: {unmatched.sum():,}, placed next to their name-group S1: {has_key.sum() / max(1, unmatched.sum()):.1%}')
    pool = RNG.permutation(np.flatnonzero(((ow == '-') & ~pd.Series(unmatched, index=ow.index)).to_numpy()))
    worlds, used = {}, 0
    for w in ('A', 'B'):
        s1w = s1[(s1_world == w).to_numpy()]
        need = max(0, int(CFG['OTHER_RATIO'] * len(s1w)) - int((ow == w).sum()))
        take = pool[used:used + need]
        used += len(take)
        ow.iloc[take] = w
        othw = oth[(ow == w).to_numpy()]
        pw = pairs[pairs.s1.isin(s1w.key) & pairs.o.isin(othw.key)]
        worlds[w] = (s1w.reset_index(drop=True), othw.reset_index(drop=True), pw.reset_index(drop=True))
        log(f'world {w}: S1={len(s1w):,} others={len(othw):,} ratio={len(othw) / max(1, len(s1w)):.2f} '
            f'gt pairs={len(pw):,} (+{len(take):,} orphan distractors)')
    return worlds

WORLDS = make_worlds(train, gt_pairs)
GT = {w: WORLDS[w][2] for w in WORLDS}
del train
gc.collect()

test = load_split('test', TEST_DIR)
TEST_S1_IDS = test.loc[test.src == 1, 'entity_id'].to_numpy()   # every one of these must be in the output
if SMOKE:
    test = test[h01(test.key.values) < CFG['SMOKE_FRAC']].reset_index(drop=True)
WORLDS['T'] = (test[test.src == 1].reset_index(drop=True), test[test.src != 1].reset_index(drop=True), None)
del test
gc.collect()
log('test:', {c: int(n) for c, n in WORLDS['T'][0].country.value_counts().items()}, 'S1 by country')

[   2.7 min] unmatched records: 2,681,854, placed next to their name-group S1: 94.2%
[   2.7 min] world A: S1=585,020 others=3,369,715 ratio=5.76 gt pairs=2,023,586 (+650,438 orphan distractors)
[   2.8 min] world B: S1=249,021 others=1,434,360 ratio=5.76 gt pairs=863,214 (+275,455 orphan distractors)
[   3.6 min] test: {'India': 809986, 'US': 663106, 'France': 259452} S1 by country


## 3. Normalisation, with maps learned from world A only
* **Transliteration.** In matched pairs the Indic-script name usually has the same number of tokens as its Latin S1 name. We align them position by position and keep the majority mapping (`प्राइवेट → private`, `ராஜ் → raj`). Whole address components are aligned the same way (`हरियाणा → haryana`). Unseen tokens fall back to `unidecode`.
* **Abbreviations and typos.** When a matched pair differs by exactly one token and the short token is a subsequence of the long one (`tx → texas`, `pvt → private`), we keep the substitution if it explains ≥60% of that short token's substitutions. This filters out random typo noise.
* **Name views.** `clean` (all tokens), `core` (legal words such as LLC/Pvt/Ltd/SARL removed), `alias` (the part before `dba`), `compact` (core with spaces removed, which catches `maurewilliamscolombier.com`), and `skeleton` (vowels and `h` removed, repeats collapsed, which catches `raam ↔ ram`, `laxmi ↔ lakshmi`).
* **Address.** Street-type expansion (EN + FR, since France is unseen), `null` dropped, `45ND → 45`, and a canonical number set (`AF-0684 ↔ Af-684`; Devanagari digits parsed too).

The maps are learned **per country** from world A. An unseen country (France) gets the generic base map, and both sides of a pair are normalised the same way, so every string feature stays symmetric.

In [6]:
NATIVE = re.compile(r'[ऀ-෿]')          # all Indic scripts
DBA = re.compile(r'\s+(?:d\s*/\s*b\s*/\s*a|dba|doing business as|t\s*/\s*a|aka)\s+', re.I)
LEET = str.maketrans({'0': 'o', '1': 'l', '3': 'e', '4': 'a', '5': 's', '7': 't', '@': 'a', '$': 's'})
HAS_ALPHA, HAS_DIGIT = re.compile(r'[a-z]'), re.compile(r'[0-9@$]')
NON_ALNUM = re.compile(r'[^a-z0-9]+')
DOMAIN = re.compile(r'\.(c[o0óò]m|net|org|in|fr)\b', re.I)
PHONE = re.compile(r'\d{7,}')
ORDINAL = re.compile(r'\b(\d+)(?:st|nd|rd|th|er|e|eme|ème)\b')
NUM = re.compile(r'\d+')
LEGAL = set('''llc inc incorporated corp corporation co company ltd limited pvt private plc llp lp pc pllc
public the and of com net org sarl sas sasu eurl sa sci cie et fils ets group groupe'''.split())
# generic street types (EN + FR); 'st' is left out on purpose (street in the US, saint in France)
ADDR_BASE = {'rd': 'road', 'ave': 'avenue', 'av': 'avenue', 'dr': 'drive', 'blvd': 'boulevard', 'bd': 'boulevard',
             'ln': 'lane', 'ct': 'court', 'pl': 'place', 'ter': 'terrace', 'hwy': 'highway', 'pkwy': 'parkway',
             'cir': 'circle', 'sq': 'square', 'r': 'rue', 'rte': 'route', 'ch': 'chemin', 'imp': 'impasse',
             'all': 'allee', 'fbg': 'faubourg', 'nr': 'near', 'opp': 'opposite', 'bldg': 'building', 'apt': 'apartment'}

def ascii_fold(s):
    s = unicodedata.normalize('NFKD', s)
    return ''.join(c for c in s if not unicodedata.combining(c)).lower()

def raw_tokens(s, tmap, leet):
    out = []
    for t in s.split():
        if NATIVE.search(t):
            t = tmap.get(t) or unidecode(t).lower()
        else:
            t = ascii_fold(t)
            if not t.isascii():
                t = unidecode(t).lower()
        t = t.replace('.', '').replace("'", '')
        if leet and HAS_ALPHA.search(t) and HAS_DIGIT.search(t):
            t = t.translate(LEET)
        out.extend(x for x in NON_ALNUM.split(t) if x)
    return out

def skeleton(t):
    s = t[0] + re.sub('[aeiouyh]', '', t[1:])
    return re.sub(r'(.)\1+', r'\1', s)

def norm_name(name, tmap, amap):
    name = PHONE.sub(' ', DOMAIN.sub(' ', name))
    parts = DBA.split(name, maxsplit=1)
    toks = [amap.get(t, t) for t in raw_tokens(name, tmap, True)]
    core = [t for t in (amap.get(t, t) for t in raw_tokens(parts[-1], tmap, True)) if t not in LEGAL]
    alias = []
    if len(parts) == 2:
        alias = [t for t in (amap.get(t, t) for t in raw_tokens(parts[0], tmap, True)) if t not in LEGAL]
    return toks, core or [t for t in toks if t not in LEGAL] or toks, alias

def norm_addr(addr, tmap, amap):
    comps = []
    for c in addr.split(','):
        c = c.strip()
        if not c or c.lower() == 'null':
            continue
        if NATIVE.search(c):
            c = tmap.get(c, c)
        c = ORDINAL.sub(r'\1', ascii_fold(c))
        comps.append(' '.join(amap.get(t, t) for t in raw_tokens(c, tmap, False)))
    return [t for c in comps for t in c.split()], comps

def addr_numbers(addr):
    out = set()
    for n in NUM.findall(ORDINAL.sub(r'\1', addr)):
        try:
            out.add(str(int(n)))           # int() also parses Devanagari/Tamil digits
        except ValueError:
            pass
    return ' '.join(sorted(out))

def _subseq(a, b):
    it = iter(b)
    return all(c in it for c in a)

def learn_maps(a_texts, b_texts, kind, min_count=30, min_share=0.6):
    native, sub = Counter(), Counter()
    for a, b in zip(a_texts, b_texts):
        if NATIVE.search(b):
            if kind == 'name':
                ta, tb = a.split(), b.split()
                if len(ta) == len(tb):
                    for x, y in zip(ta, tb):
                        if NATIVE.search(y) and not NATIVE.search(x):
                            native[(y, ascii_fold(x).replace('.', ''))] += 1
            else:
                ca, cb = {c.strip() for c in a.split(',')}, {c.strip() for c in b.split(',')}
                da, db = ca - cb, cb - ca
                if len(da) == 1 and len(db) == 1:
                    (x,), (y,) = da, db
                    if NATIVE.search(y):
                        native[(y, ascii_fold(x))] += 1
            continue
        sa, sb = set(raw_tokens(a, {}, kind == 'name')), set(raw_tokens(b, {}, kind == 'name'))
        da, db = sa - sb, sb - sa
        if len(da) == 1 and len(db) == 1:
            (x,), (y,) = da, db
            short, long_ = sorted((x, y), key=len)
            if short != long_ and short[0] == long_[0] and _subseq(short, long_) and not short.isdigit():
                sub[(short, long_)] += 1

    def best(counter, share):
        tot = Counter()
        for (k, _), n in counter.items():
            tot[k] += n
        by = {}
        for (k, v), n in counter.items():
            if n >= min_count and (not share or n >= min_share * tot[k]) and n > by.get(k, ('', 0))[1]:
                by[k] = (v, n)
        return {k: v for k, (v, _) in by.items()}
    return best(native, False), best(sub, True)

def fit_maps(s1, oth, pairs):
    a = s1.set_index('key').loc[pairs.s1.values]
    b = oth.set_index('key').loc[pairs.o.values]
    maps = {}
    for c in a.country.unique():
        m = (a.country == c).to_numpy()
        tn, an = learn_maps(a.business_name.values[m], b.business_name.values[m], 'name')
        ta, aa = learn_maps(a.business_address.values[m], b.business_address.values[m], 'addr')
        maps[c] = ({**ta, **tn}, an, {**ADDR_BASE, **aa})
        log(f'maps[{c}]: {len(tn)} native name tokens, {len(ta)} native addr comps, '
            f'{len(an)} name subs, {len(aa)} addr subs')
    return maps

MAPS_DEFAULT = ({}, {}, ADDR_BASE)
MAPS = fit_maps(*WORLDS['A'])

# self-check on the tricky cases
_t, _c, _al = norm_name('Arcsolx dba Kumar & Co', {}, {})
assert _c == ['kumar'] and _al == ['arcsolx'], (_c, _al)
assert norm_name('Pink Chemica1s Ltd', {}, {})[1] == ['pink', 'chemicals']
assert norm_name('maurewilliamscolombier.com', {}, {})[1] == ['maurewilliamscolombier']
assert norm_addr('Nord, null, 8 R. DE BOUVINES', {}, ADDR_BASE)[0] == ['nord', '8', 'rue', 'de', 'bouvines']
assert norm_addr('45ND TERRACE', {}, ADDR_BASE)[0] == ['45', 'terrace']
assert addr_numbers('AF-0684, 9487203') == addr_numbers('Af-684, 9487203')
print('normalisation self-check ok; example:', MAPS.get('US', MAPS_DEFAULT)[2].get('tx'), MAPS.get('US', MAPS_DEFAULT)[1].get('pvt'))

[   4.6 min] maps[US]: 0 native name tokens, 0 native addr comps, 22 name subs, 75 addr subs
[   5.4 min] maps[India]: 737 native name tokens, 16 native addr comps, 26 name subs, 12 addr subs
normalisation self-check ok; example: texas None


In [7]:
NORM_COLS = ['n_clean', 'n_core', 'n_alias', 'n_compact', 'n_skel', 'a_tok', 'a_nums', 'a_ncomp']
_MAPS = None

def _init(maps):
    global _MAPS
    _MAPS = maps

def _norm_chunk(rows):
    out = []
    for name, addr, country in rows:
        tmap, nmap, amap = _MAPS.get(country, MAPS_DEFAULT)
        toks, core, alias = norm_name(name, tmap, nmap)
        at, comps = norm_addr(addr, tmap, amap)
        out.append((' '.join(toks), ' '.join(core), ' '.join(alias), ''.join(core),
                    ' '.join(skeleton(t) for t in core), ' '.join(at), addr_numbers(addr), len(comps)))
    return out

def normalize(df):
    rows = list(zip(df.business_name.values, df.business_address.values, df.country.values))
    chunks = [rows[i:i + 20000] for i in range(0, len(rows), 20000)]
    with Pool(NCPU, initializer=_init, initargs=(MAPS,)) as p:
        res = [r for c in p.imap(_norm_chunk, chunks) for r in c]
    out = pd.DataFrame(res, columns=NORM_COLS, index=df.index)
    out['a_ncomp'] = out.a_ncomp.astype(np.int16)
    return pd.concat([df, out], axis=1)

def fname(c):
    return re.sub(r'\W', '_', c)

COUNTRIES = {}
for w, (s1, oth, pairs) in WORLDS.items():
    owner = pd.Series(pairs.s1.values, index=pairs.o.values) if pairs is not None else pd.Series(dtype=np.int64)
    COUNTRIES[w] = sorted(s1.country.unique())
    for c in COUNTRIES[w]:
        s1c = normalize(s1[s1.country == c].reset_index(drop=True))
        oc = normalize(oth[oth.country == c].reset_index(drop=True))
        pos = pd.Series(np.arange(len(s1c)), index=s1c.key.values)
        oc['owner'] = oc.key.map(owner).map(pos).fillna(-1).astype(np.int32).to_numpy()   # position of true S1, -1 = none
        s1c.to_parquet(TMP / f'{w}_{fname(c)}_s1.parquet')
        oc.to_parquet(TMP / f'{w}_{fname(c)}_o.parquet')
        log(f'normalized {w}/{c}: S1={len(s1c):,} others={len(oc):,}')
    orphan_countries = set(oth.country.unique()) - set(COUNTRIES[w])
    if orphan_countries:
        log(f'  {w}: others in countries without S1 (never matchable): {orphan_countries}')
WORLD_S1_KEYS = {w: WORLDS[w][0].key.to_numpy() for w in WORLDS}
del WORLDS
gc.collect()

def load_frames(w, c):
    return pd.read_parquet(TMP / f'{w}_{fname(c)}_s1.parquet'), pd.read_parquet(TMP / f'{w}_{fname(c)}_o.parquet')

print(load_frames('A', COUNTRIES['A'][0])[1][['business_name', 'n_core', 'n_skel', 'a_tok', 'a_nums']].head(8).to_string())

[   6.5 min] normalized A/India: S1=213,814 others=1,258,470
[   8.0 min] normalized A/US: S1=371,206 others=2,111,245
[   8.6 min] normalized B/India: S1=108,916 others=618,305
[   9.3 min] normalized B/US: S1=140,105 others=816,055
[  10.3 min] normalized T/France: S1=259,452 others=1,434,993
[  14.7 min] normalized T/India: S1=809,986 others=4,717,565
[  17.4 min] normalized T/US: S1=663,106 others=3,817,031
                           business_name                n_core       n_skel                                                                                                        a_tok    a_nums
0                रियल मॉडर्न फूड लिमिटेड     real modern foods  rl mdrn fds                                          shp no g 4 plot no 19 sector 42 nerul thane navi mumbai maharashtra   19 4 42
1      குளோபல் பிசினஸ் பிரைவேட் லிமிடெட்       global business    glbl bsns                                                                        city 21 s ambattur chennai tamil nadu        21


## 4. Fine-tuning the multilingual bi-encoder (dense view)
`intfloat/multilingual-e5-small` (MIT, 118M) already embeds Latin and Indic scripts into one space. We fine-tune it contrastively on world-A pairs `(S1 text, matched S2/S3 text)` with a symmetric InfoNCE loss:

* **One pair per S1 per epoch**, so a batch never contains two positives of the same entity (no false negatives).
* **Hard in-batch negatives for free.** Batches are formed from S1s of the same country *and the same state/region*, so the other 255 rows are nearby businesses rather than random ones.
* fp16 autocast. `DataParallel` spreads the batch over both T4s.

The fine-tuned embeddings serve twice: as the primary blocking view (GPU brute-force top-k) and as a model feature (cosine similarity).

In [8]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

def rec_text(names, addrs):
    return ['query: ' + (n + ' | ' + a).lower() for n, a in zip(names, addrs)]

class MeanPool(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m
    def forward(self, input_ids, attention_mask):
        h = self.m(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        msk = attention_mask.unsqueeze(-1).to(h.dtype)
        e = (h * msk).sum(1) / msk.sum(1).clamp(min=1)
        return torch.nn.functional.normalize(e.float(), dim=-1)

def wrap(m):
    return torch.nn.DataParallel(m) if NGPU > 1 else m

def unwrap(m):
    return m.module if isinstance(m, torch.nn.DataParallel) else m

def train_biencoder():
    a_txt, b_txt, order_key = [], [], []
    for c in COUNTRIES['A']:
        s1, oc = load_frames('A', c)
        m = oc[oc.owner >= 0].sample(frac=1, random_state=CFG['SEED']).drop_duplicates('owner')
        s = s1.iloc[m.owner.values]
        a_txt += rec_text(s.business_name, s.business_address)
        b_txt += rec_text(m.business_name, m.business_address)
        region = s.business_address.str.rsplit(',', n=1).str[-1].str.strip().str.lower()
        order_key += (c + '|' + region).tolist()
    n = min(len(a_txt), CFG['EMB_FT_PAIRS'])
    idx = RNG.permutation(len(a_txt))[:n]
    idx = idx[np.argsort(np.asarray(order_key, dtype=object)[idx], kind='stable')]   # same-region batches
    bs = CFG['EMB_BS']
    batches = [idx[i:i + bs] for i in range(0, n - bs + 1, bs)] or [idx]
    tok = AutoTokenizer.from_pretrained(CFG['EMB_MODEL'])
    model = wrap(MeanPool(AutoModel.from_pretrained(CFG['EMB_MODEL'])).to(DEVICE))
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['EMB_LR'], weight_decay=0.01)
    steps = len(batches) * CFG['EMB_EPOCHS']
    sched = get_linear_schedule_with_warmup(opt, int(0.05 * steps), steps)
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE == 'cuda')
    lossf = torch.nn.CrossEntropyLoss()
    model.train()
    step = 0
    for ep in range(CFG['EMB_EPOCHS']):
        for bi in RNG.permutation(len(batches)):
            b = batches[bi]
            ta = tok([a_txt[i] for i in b], padding=True, truncation=True, max_length=CFG['EMB_MAXLEN'], return_tensors='pt').to(DEVICE)
            tb = tok([b_txt[i] for i in b], padding=True, truncation=True, max_length=CFG['EMB_MAXLEN'], return_tensors='pt').to(DEVICE)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
                ea = model(ta['input_ids'], ta['attention_mask'])
                eb = model(tb['input_ids'], tb['attention_mask'])
            logits = ea @ eb.T / CFG['EMB_TEMP']
            lab = torch.arange(len(b), device=logits.device)
            loss = (lossf(logits, lab) + lossf(logits.T, lab)) / 2
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            step += 1
            if step % 200 == 0 or step == steps:
                log(f'bi-encoder step {step}/{steps} loss {loss.item():.4f}')
    model.eval()
    return tok, model

@torch.no_grad()
def encode(tok, model, texts, bs=1024):
    order = np.argsort([len(t) for t in texts])         # length bucketing = far less padding
    out = np.empty((len(texts), unwrap(model).m.config.hidden_size), dtype=np.float16)
    for i in range(0, len(texts), bs):
        ix = order[i:i + bs]
        t = tok([texts[j] for j in ix], padding=True, truncation=True, max_length=CFG['EMB_MAXLEN'], return_tensors='pt').to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
            out[ix] = model(t['input_ids'], t['attention_mask']).cpu().numpy().astype(np.float16)
    return out

BI_TOK, BI_MODEL = train_biencoder()
log('bi-encoder trained')
for w in COUNTRIES:
    for c in COUNTRIES[w]:
        s1, oc = load_frames(w, c)
        np.save(TMP / f'{w}_{fname(c)}_s1.npy', encode(BI_TOK, BI_MODEL, rec_text(s1.business_name, s1.business_address)))
        np.save(TMP / f'{w}_{fname(c)}_o.npy', encode(BI_TOK, BI_MODEL, rec_text(oc.business_name, oc.business_address)))
        log(f'embedded {w}/{c}')
unwrap(BI_MODEL).m.save_pretrained(WORK / 'biencoder')
BI_TOK.save_pretrained(WORK / 'biencoder')
del BI_MODEL
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[  19.8 min] bi-encoder step 200/2156 loss 0.0088
[  21.2 min] bi-encoder step 400/2156 loss 0.0027
[  22.7 min] bi-encoder step 600/2156 loss 0.0048
[  24.1 min] bi-encoder step 800/2156 loss 0.0021
[  25.6 min] bi-encoder step 1000/2156 loss 0.0030
[  27.0 min] bi-encoder step 1200/2156 loss 0.0080
[  28.5 min] bi-encoder step 1400/2156 loss 0.0029
[  29.9 min] bi-encoder step 1600/2156 loss 0.0008
[  31.4 min] bi-encoder step 1800/2156 loss 0.0038
[  32.8 min] bi-encoder step 2000/2156 loss 0.0041
[  34.0 min] bi-encoder step 2156/2156 loss 0.0162
[  34.0 min] bi-encoder trained
[  39.7 min] embedded A/India
[  47.6 min] embedded A/US
[  50.4 min] embedded B/India
[  53.4 min] embedded B/US
[  58.7 min] embedded T/France
[  81.4 min] embedded T/India
[  96.2 min] embedded T/US


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 5. Blocking: four views, a union, exact re-scoring
Each S2/S3 record queries the S1 index **of its own country**. We query from the other side because an other belongs to at most one S1, so its top-k is a small, natural candidate list.

| view | text | catches |
|---|---|---|
| `dense` top-8 | fine-tuned e5, FAISS IVF (probes 2% of clusters) | cross-script names, reordering, general noise |
| `nchar` top-4 | TF-IDF char 3-grams of the compact core name | typos, domains, missing spaces |
| `nword` top-4 | TF-IDF words of the core name | rare-word hits (`Orellana`) |
| `aword` top-4 | TF-IDF words of the address, numbers included | trade names that share nothing but the address (`Dréxkor`), PIN/house numbers |

Very common tokens are dropped from the sparse views (`max_df` 0.5–1%). Their posting lists are huge, they made blocking quadratic, and their IDF weight is tiny anyway. After the union, **every view is recomputed exactly for every pair** (row-wise dot products), so a pair found only by `aword` still gets its true `dense` and `nchar` scores. Then we add group context: rank within the other's candidate list, gap to the best candidate, top-1 vs top-2 gap, candidate count, and S1-side in-degree / rank. The ≤1-owner structure makes these the strongest cheap features.

In [9]:
SPARSE_VIEWS = {
    'nchar': ('n_compact', dict(analyzer='char', ngram_range=(3, 3), min_df=2, max_df=0.01)),
    'nword': ('n_core', dict(analyzer='word', token_pattern=r'\S+', min_df=1, max_df=0.01)),
    'aword': ('a_tok', dict(analyzer='word', token_pattern=r'\S+', min_df=2, max_df=0.005)),
}
VIEWS = ['dense'] + list(SPARSE_VIEWS)

def sparse_view(s1, oc, col, kw, k):
    vec = TfidfVectorizer(dtype=np.float32, sublinear_tf=True, **kw)
    vec.fit(pd.concat([s1[col], oc[col]]))                   # unlabeled, transductive IDF is fair
    X, Q = vec.transform(s1[col]), vec.transform(oc[col])
    XT = X.T.tocsr()
    rows, cols = [], []
    for st in range(0, Q.shape[0], 200_000):
        r = sp_matmul_topn(Q[st:st + 200_000], XT, top_n=k, threshold=CFG['SPARSE_MIN'], n_threads=NCPU).tocoo()
        rows.append(r.row + st)
        cols.append(r.col)
    return X, Q, np.concatenate(rows), np.concatenate(cols)

def dense_topk(Es, Eo, k):
    xs = np.ascontiguousarray(Es, dtype=np.float32)
    d, k = xs.shape[1], min(k, len(xs))
    if len(xs) < CFG['IVF_MIN']:
        index = faiss.IndexFlatIP(d)
    else:
        nlist = int(4 * np.sqrt(len(xs)))
        index = faiss.IndexIVFFlat(faiss.IndexFlatIP(d), d, nlist, faiss.METRIC_INNER_PRODUCT)
        index.train(xs[RNG.choice(len(xs), min(len(xs), 40 * nlist), replace=False)])
        index.nprobe = max(8, int(nlist * CFG['NPROBE_FRAC']))
    index.add(xs)
    rows, cols = [], []
    for st in range(0, len(Eo), 500_000):
        _, I = index.search(np.ascontiguousarray(Eo[st:st + 500_000], dtype=np.float32), k)
        r = np.repeat(np.arange(st, st + len(I)), k)
        c = I.ravel()
        m = c >= 0
        rows.append(r[m]); cols.append(c[m])
    del index, xs
    return np.concatenate(rows), np.concatenate(cols).astype(np.int64)

def dense_rowdot(Es, Eo, si, oi, chunk=400_000):
    out = np.empty(len(si), np.float32)
    for st in range(0, len(si), chunk):
        a = torch.from_numpy(Es[si[st:st + chunk]]).to(DEVICE).float()
        b = torch.from_numpy(Eo[oi[st:st + chunk]]).to(DEVICE).float()
        out[st:st + chunk] = (a * b).sum(1).cpu().numpy()
        del a, b
    torch.cuda.empty_cache()
    return out

def sparse_rowdot(Q, X, oi, si, chunk=1_000_000):
    out = np.empty(len(oi), np.float32)
    for st in range(0, len(oi), chunk):
        out[st:st + chunk] = np.asarray(Q[oi[st:st + chunk]].multiply(X[si[st:st + chunk]]).sum(1)).ravel()
    return out

def group_stats(g, x):
    # per row: 0-based descending rank of x within group g, group top1, group top2 (nan if single), group size
    order = np.lexsort((-x, g))
    gs, xs = g[order], x[order]
    start = np.r_[True, gs[1:] != gs[:-1]]
    first = np.flatnonzero(start)
    gp = np.cumsum(start) - 1
    size = np.diff(np.r_[first, len(g)])
    top2 = np.where(size > 1, xs[np.minimum(first + 1, len(g) - 1)], np.nan)
    rank, t1, t2, sz = (np.empty(len(g), np.float32) for _ in range(4))
    rank[order] = np.arange(len(g)) - first[gp]
    t1[order] = xs[first][gp]
    t2[order] = top2[gp]
    sz[order] = size[gp]
    return rank, t1, t2, sz

def block_country(w, c):
    s1, oc = load_frames(w, c)
    Es = np.load(TMP / f'{w}_{fname(c)}_s1.npy')
    Eo = np.load(TMP / f'{w}_{fname(c)}_o.npy')
    hit_o, hit_s = [], []
    r, cc = dense_topk(Es, Eo, CFG['K_DENSE'])
    hit_o.append(r); hit_s.append(cc)
    mats = {}
    for v, (col, kw) in SPARSE_VIEWS.items():
        try:
            X, Q, r, cc = sparse_view(s1, oc, col, kw, CFG['K_SPARSE'])
        except ValueError:                                   # empty vocabulary (tiny smoke data)
            mats[v] = None
            continue
        mats[v] = (X, Q)
        hit_o.append(r); hit_s.append(cc)
    ns = len(s1)
    key = np.unique(np.concatenate(hit_o).astype(np.int64) * ns + np.concatenate(hit_s))
    oi, si = (key // ns).astype(np.int32), (key % ns).astype(np.int32)   # sorted by oi
    del key, hit_o, hit_s
    n = len(oi)
    X = np.empty((n, len(F1_COLS)), np.float32, order='F')   # column-major: every column write is contiguous
    J = {name: j for j, name in enumerate(F1_COLS)}
    def put(name, arr):
        X[:, J[name]] = arr
    put('dense', dense_rowdot(Es, Eo, si, oi))
    for v in SPARSE_VIEWS:
        put(v, sparse_rowdot(mats[v][1], mats[v][0], oi, si) if mats[v] else 0.0)
    del mats, Es, Eo
    gc.collect()
    for v in VIEWS:
        x = X[:, J[v]]
        rk, t1, t2, sz = group_stats(oi, x)
        put(f'{v}_rk', rk); put(f'{v}_gap', t1 - x); put(f'{v}_t12', t1 - np.nan_to_num(t2, nan=0.0))
        del rk, t1, t2
    put('n_cand', sz)
    d = X[:, J['dense']]
    rk, t1, _, sz = group_stats(si, d)
    put('s_rk', rk); put('s_gap', t1 - d); put('s_indeg', sz)
    del rk, t1, sz
    put('s_top1cnt', np.bincount(si, weights=(X[:, J['dense_rk']] == 0), minlength=ns)[si])
    put('o_src', oc.src.values[oi])
    put('o_addr_empty', oc.a_tok.values[oi] == '')
    put('o_native', oc.business_name.str.contains(NATIVE).values[oi])
    put('o_ntok', oc.n_core.str.count(' ').values[oi] + 1)
    put('s_ntok', s1.n_core.str.count(' ').values[si] + 1)
    y = (oc.owner.values[oi] == si).astype(np.int8)
    fold = (h01(oc.key.values) < 0.5).astype(np.int8)[oi]
    n_gt = int((oc.owner >= 0).sum())
    STATS.setdefault(w, Counter()).update(gt=n_gt, found=int(y.sum()), pairs=n, others=len(oc))
    log(f'blocked {w}/{c}: {n:,} pairs ({n / max(1, len(oc)):.1f}/other), '
        f'recall {y.sum() / max(1, n_gt):.4f}' if n_gt else f'blocked {w}/{c}: {n:,} pairs')
    return dict(oi=oi, si=si, X=X, y=y, fold=fold)

def to_frame(B, idx, p1):
    # small DataFrame of the kept (pruned) rows for stage 2
    F = pd.DataFrame(B['X'][idx], columns=F1_COLS)
    for k in ('oi', 'si', 'y', 'fold'):
        F[k] = B[k][idx]
    F['p1'] = p1[idx]
    return F

F1_COLS = VIEWS + [f'{v}_{s}' for v in VIEWS for s in ('rk', 'gap', 't12')] + \
    ['n_cand', 's_rk', 's_gap', 's_indeg', 's_top1cnt', 'o_src', 'o_addr_empty', 'o_native', 'o_ntok', 's_ntok']
STATS = {}

## 6. Stage 1: a cheap XGBoost pruner, which produces `candidate_pairs.tsv`
Trained on world A with **2-fold out-of-fold predictions** (folds split by S2/S3 record), so the pruning that stage 2 is trained on looks exactly like the pruning applied to B and test. For each other we keep the **top-3 S1 candidates with p ≥ 0.005**. This pruned set is what the final model runs inference on, so, per the rules, it is what we submit as `candidate_pairs.tsv`. B and test are scored by the average of the two fold models.

In [10]:
XGB1 = dict(objective='binary:logistic', eval_metric='logloss', tree_method='hist', device=XGB_DEV,
            max_depth=8, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
            max_bin=256, seed=CFG['SEED'])

def fit_xgb(params, Xtr, ytr, Xva, yva, rounds):
    dtr = xgb.QuantileDMatrix(Xtr, ytr)
    dva = xgb.QuantileDMatrix(Xva, yva, ref=dtr)
    return xgb.train(params, dtr, rounds, evals=[(dva, 'va')], early_stopping_rounds=50, verbose_eval=100)

def predict(bst, X, cols=None, chunk=3_000_000):
    # X: ndarray, or DataFrame + cols (converted chunk by chunk to keep peak RAM low)
    out = np.empty(len(X), np.float32)
    for st in range(0, len(X), chunk):
        xc = X.iloc[st:st + chunk][cols].to_numpy(np.float32) if cols else X[st:st + chunk]
        out[st:st + chunk] = bst.predict(xgb.DMatrix(xc), iteration_range=(0, bst.best_iteration + 1))
    return out

def prune_idx(oi, p1):
    rk = group_stats(oi, p1)[0]
    return np.flatnonzero((rk < CFG['KEEP_PER_OTHER']) & (p1 >= CFG['STAGE1_MIN_P']))

A1 = {c: block_country('A', c) for c in COUNTRIES['A']}
sizes = [len(A1[c]['oi']) for c in COUNTRIES['A']]
# No full concatenation: joining 68M rows needed two copies at once and ran Kaggle out of RAM.
# Each fold trains on a uniform row sample (keeps calibration); OOF predictions go country by country.
def fold_sample(f, train, cap):
    tot = sum(int(((A1[c]['fold'] != f) if train else (A1[c]['fold'] == f)).sum()) for c in COUNTRIES['A'])
    rate = min(1.0, cap / max(1, tot))
    xs, ys = [], []
    for c in COUNTRIES['A']:
        m = (A1[c]['fold'] != f) if train else (A1[c]['fold'] == f)
        idx = np.flatnonzero(m)
        if rate < 1:
            idx = idx[RNG.random(len(idx)) < rate]
        xs.append(A1[c]['X'][idx]); ys.append(A1[c]['y'][idx])
    return np.concatenate(xs), np.concatenate(ys)

def predict_rows(bst, X, idx, chunk=3_000_000):
    out = np.empty(len(idx), np.float32)
    for st in range(0, len(idx), chunk):
        out[st:st + chunk] = predict(bst, X[idx[st:st + chunk]])
    return out

STAGE1 = []
for f in (0, 1):
    Xtr, ytr = fold_sample(f, True, CFG['S1_TRAIN_ROWS'])
    Xva, yva = fold_sample(f, False, 2_000_000)
    bst = fit_xgb(XGB1, Xtr, ytr, Xva, yva, 600)
    del Xtr, ytr, Xva, yva
    gc.collect()
    STAGE1.append(bst)
    bst.save_model(WORK / f'stage1_fold{f}.json')
parts = []
for c in COUNTRIES['A']:
    B = A1[c]
    p1_oof = np.empty(len(B['oi']), np.float32)
    for f in (0, 1):
        idx = np.flatnonzero(B['fold'] == f)
        p1_oof[idx] = predict_rows(STAGE1[f], B['X'], idx)     # model f never saw fold f
    parts.append(to_frame(B, prune_idx(B['oi'], p1_oof), p1_oof).assign(country=c))
    del B['X'], B, p1_oof
    gc.collect()
n_all = sum(sizes)
del A1
FA2 = pd.concat(parts, ignore_index=True)
del parts
gc.collect()
log(f'stage-1 A: {n_all:,} -> {len(FA2):,} pairs; recall of GT kept: {FA2.y.sum() / STATS["A"]["gt"]:.4f} '
    f'(blocking recall {STATS["A"]["found"] / STATS["A"]["gt"]:.4f})')
imp = pd.Series(STAGE1[0].get_score(importance_type='gain')).sort_values(ascending=False)
imp.index = [F1_COLS[int(i[1:])] if i.startswith('f') and i[1:].isdigit() else i for i in imp.index]
print(imp.head(15).round(1).to_string())

[ 105.5 min] blocked A/India: 20,792,540 pairs (16.5/other), recall 0.9894
[ 130.3 min] blocked A/US: 34,458,934 pairs (16.3/other), recall 0.9906
[0]	va-logloss:0.08042
[100]	va-logloss:0.01173
[200]	va-logloss:0.01117
[300]	va-logloss:0.01100
[400]	va-logloss:0.01093
[500]	va-logloss:0.01088
[599]	va-logloss:0.01086
[0]	va-logloss:0.08032
[100]	va-logloss:0.01174
[200]	va-logloss:0.01114
[300]	va-logloss:0.01097
[400]	va-logloss:0.01090
[500]	va-logloss:0.01085
[599]	va-logloss:0.01083
[ 136.4 min] stage-1 A: 55,251,474 -> 2,720,548 pairs; recall of GT kept: 0.9866 (blocking recall 0.9902)
dense           5271.0
dense_rk        2569.4
dense_gap        344.0
o_addr_empty     157.4
o_native         156.8
aword            143.1
s_rk             139.2
o_ntok            93.9
s_ntok            67.8
nchar             67.0
nword_gap         66.8
nword             57.9
s_gap             48.7
aword_gap         45.6
s_top1cnt         33.5


## 7. Stage 2 features: fuzzy strings, numbers, context
Computed only on the pruned pairs (≤3 per other), with `rapidfuzz.process.cpdist`, which is vectorised, multi-threaded C++ over aligned lists.

* **Name:** ratio / token-sort / token-set / partial / Jaro-Winkler on the core name, token-set on the full name (legal suffixes still in), ratio + partial on the compact name (domains), ratio on the skeleton (transliteration), and best DBA-alias match in both directions.
* **Address:** ratio / token-set / token-sort / partial, number-set Jaccard, first-number equality, shared long number (PIN / ZIP / phone), component-count difference, empty flags.
* **Sibling detection:** numbers in the record but not the S1 and vice versa, a *substitution* flag (a number both missing and added, e.g. `G No 02 → 15`), descriptor words added or dropped, and **group counts**. The group counts say how many of this S1's candidates share the same wrong number or the same extra words (siblings come in groups of 2–4), and how many agree cleanly with the S1. There is also an out-of-fold **log-odds of the added word**: noise adds *Center/Services* to true matches, siblings add *Group/Industries/Textiles*.
* **Context on the stage-1 probability:** rank / gap / top-2 gap within the other; rank, count, max and sum within the S1 (how crowded that S1's cluster is, which matters a lot for singletons).

In [11]:
def cp(a, b, scorer):
    return process.cpdist(a, b, scorer=scorer, workers=-1, dtype=np.float32)

def num_feats(A, B):
    n = len(A)
    jac, first, longm = (np.full(n, -1.0, np.float32) for _ in range(3))
    for i, (x, z) in enumerate(zip(A, B)):
        if x and z:
            sx, sz = x.split(), z.split()
            X_, Z_ = set(sx), set(sz)
            inter = len(X_ & Z_)
            jac[i] = inter / len(X_ | Z_)
            first[i] = float(bool(X_ & {sz[0]}) or bool(Z_ & {sx[0]}))
            lx = {t for t in X_ if len(t) >= 5}
            lz = {t for t in Z_ if len(t) >= 5}
            longm[i] = float(bool(lx & lz)) if (lx and lz) else -1.0
    return jac, first, longm

def diff_feats(A, B):
    # tokens in the other but not the S1 (extra) and vice versa (missing), plus a signature of both
    n = len(A)
    ex, mi = np.zeros(n, np.float32), np.zeros(n, np.float32)
    exs, sig = [None] * n, [None] * n
    for i, (x, z) in enumerate(zip(A, B)):
        X_, Z_ = set(x.split()), set(z.split())
        e, m = Z_ - X_, X_ - Z_
        ex[i], mi[i] = len(e), len(m)
        exs[i] = ' '.join(sorted(e))
        sig[i] = exs[i] + '|' + ' '.join(sorted(m))
    return ex, mi, exs, sig

def group_count(si, labels):
    # how many OTHER candidates of the same S1 share this label (siblings come in groups)
    if len(si) == 0:
        return np.zeros(0, np.float32)
    codes = pd.factorize(pd.Series(labels))[0].astype(np.int64)
    k = si.astype(np.int64) * (codes.max() + 1) + codes
    _, inv, cnt = np.unique(k, return_inverse=True, return_counts=True)
    return (cnt[inv] - 1).astype(np.float32)

def tok_table(xs, y):
    pos, neg = Counter(), Counter()
    for s, t in zip(xs, y):
        if s:
            (pos if t else neg).update(s.split())
    return pos, neg

def tok_lr(xs, tab):
    # most "sibling-like" added name word: noise adds Center/Services to true matches, siblings add Group/Industries
    pos, neg = tab
    out = np.full(len(xs), np.nan, np.float32)
    for i, s in enumerate(xs):
        if s:
            out[i] = min(math.log((pos[t] + 1) / (neg[t] + 1)) for t in s.split())
    return out

def stage2_features(F, s1, oc):
    oi, si = F.oi.to_numpy(), F.si.to_numpy()
    S = lambda col: s1[col].values[si].tolist()
    O = lambda col: oc[col].values[oi].tolist()
    a, b = S('n_core'), O('n_core')
    F['n_ratio'] = cp(a, b, fuzz.ratio)
    F['n_tsort'] = cp(a, b, fuzz.token_sort_ratio)
    F['n_tset'] = cp(a, b, fuzz.token_set_ratio)
    F['n_partial'] = cp(a, b, fuzz.partial_ratio)
    F['n_jw'] = cp(a, b, JaroWinkler.normalized_similarity)
    F['n_full_tset'] = cp(S('n_clean'), O('n_clean'), fuzz.token_set_ratio)
    ca, cb = S('n_compact'), O('n_compact')
    F['n_compact'] = cp(ca, cb, fuzz.ratio)
    F['n_compact_partial'] = cp(ca, cb, fuzz.partial_ratio)
    F['n_skel'] = cp(S('n_skel'), O('n_skel'), fuzz.ratio)
    sa, oa = S('n_alias'), O('n_alias')
    al1 = np.where(np.asarray(oa, dtype=object) != '', cp(a, oa, fuzz.token_set_ratio), -1)
    al2 = np.where(np.asarray(sa, dtype=object) != '', cp(sa, b, fuzz.token_set_ratio), -1)
    F['n_alias'] = np.maximum(al1, al2).astype(np.float32)
    F['n_len_diff'] = np.abs(np.array([len(x) for x in a]) - np.array([len(x) for x in b])).astype(np.float32)
    a, b = S('a_tok'), O('a_tok')
    F['a_ratio'] = cp(a, b, fuzz.ratio)
    F['a_tset'] = cp(a, b, fuzz.token_set_ratio)
    F['a_tsort'] = cp(a, b, fuzz.token_sort_ratio)
    F['a_partial'] = cp(a, b, fuzz.partial_ratio)
    F['a_num_jac'], F['a_num_first'], F['a_num_long'] = num_feats(S('a_nums'), O('a_nums'))
    F['a_ncomp_diff'] = (s1.a_ncomp.values[si].astype(np.float32) - oc.a_ncomp.values[oi])
    ns = int(si.max()) + 1 if len(si) else 1
    ex, mi, _, sig = diff_feats(S('a_nums'), O('a_nums'))
    F['num_extra'], F['num_miss'] = ex, mi
    F['num_subst'] = ((ex > 0) & (mi > 0)).astype(np.float32)
    F['num_sig_cnt'] = group_count(si, sig)
    F['s_num_clean'] = np.bincount(si, weights=(ex == 0) & (mi == 0), minlength=ns)[si].astype(np.float32)
    ex, mi, exs, sig = diff_feats(S('n_core'), O('n_core'))
    F['tok_extra'], F['tok_miss'] = ex, mi
    F['tok_sig_cnt'] = group_count(si, sig)
    F['s_tok_clean'] = np.bincount(si, weights=(ex == 0) & (mi == 0), minlength=ns)[si].astype(np.float32)
    F['x_tok'] = exs
    p1 = F.p1.to_numpy()
    rk, t1, t2, sz = group_stats(oi, p1)
    F['p1_rk'], F['p1_gap'], F['p1_t12'], F['p1_n'] = rk, t1 - p1, t1 - np.nan_to_num(t2, nan=0.0), sz
    rk, t1, _, sz = group_stats(si, p1)
    F['p1_srk'], F['p1_sgap'], F['p1_sn'] = rk, t1 - p1, sz
    F['p1_ssum'] = np.bincount(si, weights=p1)[si].astype(np.float32)
    return F

F2_COLS = F1_COLS + ['p1', 'n_ratio', 'n_tsort', 'n_tset', 'n_partial', 'n_jw', 'n_full_tset', 'n_compact',
                     'n_compact_partial', 'n_skel', 'n_alias', 'n_len_diff', 'a_ratio', 'a_tset', 'a_tsort',
                     'a_partial', 'a_num_jac', 'a_num_first', 'a_num_long', 'a_ncomp_diff',
                     'p1_rk', 'p1_gap', 'p1_t12', 'p1_n', 'p1_srk', 'p1_sgap', 'p1_sn', 'p1_ssum',
                     'num_extra', 'num_miss', 'num_subst', 'num_sig_cnt', 's_num_clean',
                     'tok_extra', 'tok_miss', 'tok_sig_cnt', 's_tok_clean', 'tok_lr']

parts = []
for c in COUNTRIES['A']:
    s1, oc = load_frames('A', c)
    Fc = FA2[FA2.country == c].reset_index(drop=True)
    Fc = stage2_features(Fc, s1, oc)
    Fc['s1'] = s1.key.values[Fc.si.values]
    Fc['o'] = oc.key.values[Fc.oi.values]
    parts.append(Fc)
FA2 = pd.concat(parts, ignore_index=True)
del parts
gc.collect()
# out-of-fold word log-odds for A (same folds as stage 1), full-A table for B / test
FA2['tok_lr'] = np.float32(np.nan)
for f in (0, 1):
    m = (FA2.fold == f).to_numpy()
    FA2.loc[m, 'tok_lr'] = tok_lr(FA2.x_tok.values[m], tok_table(FA2.x_tok.values[~m], FA2.y.values[~m]))
TOK_TAB = tok_table(FA2.x_tok.values, FA2.y.values)
log(f'stage-2 features A: {FA2.shape}; sibling-style pairs (number substitution): '
    f'{FA2.num_subst.mean():.1%} of candidates, positive rate among them {FA2.y[FA2.num_subst == 1].mean():.2f}')

[ 137.6 min] stage-2 features A: (2720548, 72); sibling-style pairs (number substitution): 24.3% of candidates, positive rate among them 0.23


## 8. Stage 2: the main XGBoost matcher
Deeper trees and a lower learning rate. Early stopping uses a 10% slice of world A split by S2/S3 record. World B is **not touched** here: it is reserved for choosing the decision rule and reporting an honest score.

In [12]:
XGB2 = dict(XGB1, max_depth=10, learning_rate=0.05, min_child_weight=10, colsample_bytree=0.7, reg_lambda=2.0)
va = h01(FA2.o.values) < 0.1
X2 = FA2[F2_COLS].to_numpy(np.float32)
STAGE2 = []
for k in range(CFG['N_STAGE2']):
    bst = fit_xgb(dict(XGB2, seed=CFG['SEED'] + k, colsample_bytree=0.6 + 0.1 * (k % 3)),
                  X2[~va], FA2.y.values[~va], X2[va], FA2.y.values[va], 3000)
    bst.save_model(WORK / f'stage2_seed{k}.json')
    STAGE2.append(bst)
del X2
imp = pd.Series(STAGE2[0].get_score(importance_type='gain')).sort_values(ascending=False)
imp.index = [F2_COLS[int(i[1:])] if i.startswith('f') and i[1:].isdigit() else i for i in imp.index]
print(imp.head(25).round(1).to_string())

[0]	va-logloss:0.53745
[100]	va-logloss:0.06091
[200]	va-logloss:0.05401
[300]	va-logloss:0.05205
[400]	va-logloss:0.05114
[500]	va-logloss:0.05055
[600]	va-logloss:0.05019
[700]	va-logloss:0.04987
[800]	va-logloss:0.04970
[900]	va-logloss:0.04954
[1000]	va-logloss:0.04943
[1100]	va-logloss:0.04934
[1200]	va-logloss:0.04928
[1300]	va-logloss:0.04926
[1400]	va-logloss:0.04926
[1408]	va-logloss:0.04925
[0]	va-logloss:0.53928
[100]	va-logloss:0.06034
[200]	va-logloss:0.05403
[300]	va-logloss:0.05206
[400]	va-logloss:0.05114
[500]	va-logloss:0.05060
[600]	va-logloss:0.05020
[700]	va-logloss:0.04991
[800]	va-logloss:0.04974
[900]	va-logloss:0.04958
[1000]	va-logloss:0.04949
[1100]	va-logloss:0.04943
[1200]	va-logloss:0.04940
[1300]	va-logloss:0.04937
[1400]	va-logloss:0.04933
[1425]	va-logloss:0.04934
[0]	va-logloss:0.53761
[100]	va-logloss:0.06048
[200]	va-logloss:0.05419
[300]	va-logloss:0.05223
[400]	va-logloss:0.05123
[500]	va-logloss:0.05066
[600]	va-logloss:0.05025
[700]	va-logloss:0.

## 8b. Set-attention reranker: the candidates of one S1 look at each other
Stage 2 (and the cross-encoder) scores one pair at a time. Siblings are defined by the *other* candidates, though. "This record and two others share house number 15, while four records agree with the S1's 02" is a statement about the whole candidate set. So each S1's candidates, up to 32 ordered by p1 and each described by the stage-2 feature vector, go through a small **Transformer encoder with no positional encoding**. Every candidate attends to all the others, and the model emits one match probability per candidate.

* Input: the same ~70 features stage 2 uses (sign·log1p, then standardised on world A). No text, so it is fast: training takes a few minutes and scoring 10M test pairs takes minutes.
* It is trained on world A, grouped by S1, with early stopping on 10% of the S1 groups. Its output `pset` is blended with p2 in step 11, and the blend weight `a` is chosen on world B. If it does not help, `a = 0` and nothing changes.

In [13]:
class SetRanker(torch.nn.Module):
    def __init__(self, d_in, d, layers, heads=4):
        super().__init__()
        self.inp = torch.nn.Sequential(torch.nn.Linear(d_in, d), torch.nn.GELU(), torch.nn.Linear(d, d))
        layer = torch.nn.TransformerEncoderLayer(d, heads, 4 * d, dropout=0.1, batch_first=True, norm_first=True)
        self.enc = torch.nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.out = torch.nn.Linear(d, 1)
    def forward(self, x, pad):
        return self.out(self.enc(self.inp(x), src_key_padding_mask=pad)).squeeze(-1)

def set_transform(X):
    X = np.sign(X) * np.log1p(np.abs(X))
    return np.clip(np.nan_to_num((X - SET_MU) / SET_SD, nan=0.0), -6, 6).astype(np.float32)

def set_groups(gid, p1):
    # rows of each group made contiguous, best p1 first
    order = np.lexsort((-p1, gid))
    g = gid[order]
    start = np.flatnonzero(np.r_[True, g[1:] != g[:-1]])
    lens = np.minimum(np.diff(np.r_[start, len(g)]), CFG['SET_MAXLEN'])
    return order, start, lens

def set_batches(order, start, lens, groups, bs):
    for b in range(0, len(groups), bs):
        g = groups[b:b + bs]
        ar = np.arange(lens[g].max())
        valid = ar[None, :] < lens[g][:, None]
        pos = order[np.minimum(start[g][:, None] + ar[None, :], len(order) - 1)]
        yield pos, valid

def train_set_ranker():
    X = FA2[F2_COLS].to_numpy(np.float32)
    Xs = np.sign(X) * np.log1p(np.abs(X))
    global SET_MU, SET_SD
    SET_MU, SET_SD = np.nanmean(Xs, 0), np.nanstd(Xs, 0) + 1e-6
    Xn = set_transform(X)
    del X, Xs
    y = FA2.y.to_numpy().astype(np.float32)
    order, start, lens = set_groups(FA2.s1.to_numpy(), FA2.p1.to_numpy())
    gkey = FA2.s1.to_numpy()[order[start]]
    is_va = h01(gkey) < 0.1
    tr_g, va_g = np.flatnonzero(~is_va), np.flatnonzero(is_va)
    model = SetRanker(Xn.shape[1], CFG['SET_DIM'], CFG['SET_LAYERS']).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['SET_LR'], weight_decay=0.01)
    steps = CFG['SET_EPOCHS'] * max(1, len(tr_g) // CFG['SET_BS'])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG['SET_LR'], total_steps=steps + 10)
    lossf = torch.nn.BCEWithLogitsLoss(reduction='none')
    best, best_state = 1e9, None
    for ep in range(CFG['SET_EPOCHS']):
        model.train()
        for pos, valid in set_batches(order, start, lens, RNG.permutation(tr_g), CFG['SET_BS']):
            x = torch.from_numpy(Xn[pos]).to(DEVICE)
            v = torch.from_numpy(valid).to(DEVICE)
            loss = (lossf(model(x, ~v), torch.from_numpy(y[pos]).to(DEVICE)) * v).sum() / v.sum()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
        model.eval()
        tot, n = 0.0, 0
        with torch.no_grad():
            for pos, valid in set_batches(order, start, lens, va_g, 2048):
                v = torch.from_numpy(valid).to(DEVICE)
                l = lossf(model(torch.from_numpy(Xn[pos]).to(DEVICE), ~v), torch.from_numpy(y[pos]).to(DEVICE))
                tot += float((l * v).sum()); n += int(v.sum())
        va_loss = tot / max(1, n)
        log(f'set ranker epoch {ep + 1}: val logloss {va_loss:.5f}')
        if va_loss < best:
            best, best_state = va_loss, {k: t.detach().clone() for k, t in model.state_dict().items()}
    model.load_state_dict(best_state)
    model.eval()
    return model

@torch.no_grad()
def set_predict(model, F, gid):
    Xn = set_transform(F[F2_COLS].to_numpy(np.float32))
    order, start, lens = set_groups(gid, F.p1.to_numpy())
    out = np.full(len(F), np.nan, np.float32)                 # rows beyond SET_MAXLEN keep NaN -> p2 is used
    for pos, valid in set_batches(order, start, lens, np.arange(len(start)), 4096):
        v = torch.from_numpy(valid).to(DEVICE)
        p = torch.sigmoid(model(torch.from_numpy(Xn[pos]).to(DEVICE), ~v)).cpu().numpy()
        out[pos[valid]] = p[valid]
    return out

SETR = train_set_ranker() if CFG['RUN_SET'] else None
if SETR is not None:
    torch.save({'state': SETR.state_dict(), 'mu': SET_MU, 'sd': SET_SD}, WORK / 'set_ranker.pt')

[ 141.8 min] set ranker epoch 1: val logloss 0.05536
[ 142.1 min] set ranker epoch 2: val logloss 0.04742
[ 142.5 min] set ranker epoch 3: val logloss 0.04366
[ 142.8 min] set ranker epoch 4: val logloss 0.04088
[ 143.1 min] set ranker epoch 5: val logloss 0.03994


## 9. (Optional, advanced) Cross-encoder for the ambiguous band
A GBDT only sees hand-made similarity numbers. A cross-encoder reads both records **jointly** with attention, so it can tell that `Hotel Limited Services` is not `Hotel Enterprises Limited` even though every fuzzy score is high. It is expensive, so we only apply it where the GBDT is unsure (`0.03 < p2 < 0.97`). Its training set is world-A pairs from the same uncertain zone (chosen by the out-of-fold p1) plus some easy pairs. Backbone: `xlm-roberta-base` (MIT, 278M), which is multilingual and so handles Hindi/Tamil/French.

The blend weight between p2 and the cross-encoder score is chosen on world B in step 11. If the cross-encoder does not help, the weight search returns 0 and nothing is lost. **v4 turns it on (`RUN_CE=True`)**, now that training contains realistic siblings. A time guard (`CE_DEADLINE_MIN`) skips it for any country that starts too late, so the run always finishes. At full scale it costs ~3 h (training ~2 h, scoring ~1 h), and on the smoke run it moved F0.5 by only +0.00002. Turn it on only for a second run if there's time left.

In [14]:
def pair_texts(F, s1, oc):
    si, oi = F.si.to_numpy(), F.oi.to_numpy()
    a = [f'{n} | {d}' for n, d in zip(s1.business_name.values[si], s1.business_address.values[si])]
    b = [f'{n} | {d}' for n, d in zip(oc.business_name.values[oi], oc.business_address.values[oi])]
    return a, b

class CEHead(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m
    def forward(self, input_ids, attention_mask):
        return self.m(input_ids=input_ids, attention_mask=attention_mask).logits.squeeze(-1).float()

def train_ce():
    band = (FA2.p1 > 0.02) & (FA2.p1 < 0.98)
    hard = FA2[band].sample(n=min(int(band.sum()), int(CFG['CE_TRAIN_PAIRS'] * 0.75)), random_state=1)
    easy = FA2[~band].sample(n=min(int((~band).sum()), CFG['CE_TRAIN_PAIRS'] - len(hard)), random_state=2)
    D = pd.concat([hard, easy])
    A_, B_, Y_ = [], [], []
    for c in COUNTRIES['A']:
        Dc = D[D.country == c]
        if len(Dc):
            s1, oc = load_frames('A', c)
            a, b = pair_texts(Dc, s1, oc)
            A_ += a; B_ += b; Y_ += Dc.y.tolist()
    log(f'cross-encoder train pairs: {len(Y_):,} (pos rate {np.mean(Y_):.3f})')
    tok = AutoTokenizer.from_pretrained(CFG['CE_MODEL'])
    model = wrap(CEHead(AutoModelForSequenceClassification.from_pretrained(CFG['CE_MODEL'], num_labels=1)).to(DEVICE))
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['CE_LR'], weight_decay=0.01)
    order = RNG.permutation(len(Y_))
    bs = CFG['CE_BS']
    steps = max(1, len(order) // bs)
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE == 'cuda')
    lossf = torch.nn.BCEWithLogitsLoss()
    Yt = torch.tensor(Y_, dtype=torch.float32)
    model.train()
    for st in range(steps):
        ix = order[st * bs:(st + 1) * bs]
        t = tok([A_[i] for i in ix], [B_[i] for i in ix], padding=True, truncation=True,
                max_length=CFG['CE_MAXLEN'], return_tensors='pt').to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
            logit = model(t['input_ids'], t['attention_mask'])
        loss = lossf(logit, Yt[ix].to(logit.device))
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        if (st + 1) % 500 == 0 or st + 1 == steps:
            log(f'cross-encoder step {st + 1}/{steps} loss {loss.item():.4f}')
    model.eval()
    return tok, model

@torch.no_grad()
def ce_score(tok, model, a, b, bs=256):
    order = np.argsort([len(x) + len(z) for x, z in zip(a, b)])
    out = np.empty(len(a), np.float32)
    for st in range(0, len(a), bs):
        ix = order[st:st + bs]
        t = tok([a[i] for i in ix], [b[i] for i in ix], padding=True, truncation=True,
                max_length=CFG['CE_MAXLEN'], return_tensors='pt').to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
            out[ix] = torch.sigmoid(model(t['input_ids'], t['attention_mask'])).cpu().numpy()
    return out

CE = train_ce() if CFG['RUN_CE'] else None
if CE:
    unwrap(CE[1]).m.save_pretrained(WORK / 'cross_encoder')
    CE[0].save_pretrained(WORK / 'cross_encoder')
del FA2
gc.collect()

[ 143.4 min] cross-encoder train pairs: 250,000 (pos rate 0.650)


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[ 147.5 min] cross-encoder step 500/3906 loss 0.1793
[ 151.4 min] cross-encoder step 1000/3906 loss 0.0580
[ 155.3 min] cross-encoder step 1500/3906 loss 0.2102
[ 159.1 min] cross-encoder step 2000/3906 loss 0.0895
[ 163.0 min] cross-encoder step 2500/3906 loss 0.0602
[ 166.9 min] cross-encoder step 3000/3906 loss 0.0645
[ 170.8 min] cross-encoder step 3500/3906 loss 0.0795
[ 174.0 min] cross-encoder step 3906/3906 loss 0.1114


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

309

## 10. Score world B and test with the frozen models
The same code path runs for both, country by country: block → stage-1 (average of the 2 fold models) → prune → stage-2 features → p2 → cross-encoder on the band. Only `(s1, o, p1, p2, pce)` is kept per pair, as int64 keys, to stay small.

In [15]:
def score_world(w):
    out = []
    for c in COUNTRIES[w]:
        B = block_country(w, c)
        p1 = np.mean([predict(b, B['X']) for b in STAGE1], axis=0)
        F = to_frame(B, prune_idx(B['oi'], p1), p1)
        del B, p1
        gc.collect()
        s1, oc = load_frames(w, c)
        F = stage2_features(F, s1, oc)
        F['tok_lr'] = tok_lr(F.x_tok.values, TOK_TAB)
        F['p2'] = np.mean([predict(b, F, F2_COLS) for b in STAGE2], axis=0)
        F['pset'] = set_predict(SETR, F, F.si.to_numpy()) if SETR is not None else np.float32(np.nan)
        F['pce'] = np.nan
        use_ce = CE and (time.time() - T0) / 60 < CFG['CE_DEADLINE_MIN']
        if CE and not use_ce:
            log(f'  time guard: skipping cross-encoder for {w}/{c} (p2 only)')
        if use_ce:
            band = ((F.p2 > CFG['CE_BAND'][0]) & (F.p2 < CFG['CE_BAND'][1])).to_numpy()
            if band.any():
                a, b = pair_texts(F[band], s1, oc)
                F.loc[band, 'pce'] = ce_score(*CE, a, b)
            log(f'  cross-encoder scored {band.sum():,} ambiguous pairs')
        out.append(pd.DataFrame({'s1': s1.key.values[F.si.values], 'o': oc.key.values[F.oi.values],
                                 'p1': F.p1.values, 'p2': F.p2.values, 'pce': F.pce.values.astype(np.float32),
                                 'pset': np.asarray(F.pset.values, np.float32),
                                 'y': F.y.values}))
        del F
        gc.collect()
        torch.cuda.empty_cache()
    P = pd.concat(out, ignore_index=True)
    P.to_parquet(TMP / f'scored_{w}.parquet')
    return P

PB = score_world('B')
log(f'B: blocking recall {STATS["B"]["found"] / STATS["B"]["gt"]:.4f}, '
    f'after stage-1 pruning {PB.y.sum() / STATS["B"]["gt"]:.4f}, pairs {len(PB):,}')
PT = score_world('T')
log(f'test: {len(PT):,} candidate pairs')

[ 176.8 min] blocked B/India: 10,058,248 pairs (16.3/other), recall 0.9888
[ 178.6 min]   cross-encoder scored 74,633 ambiguous pairs
[ 182.9 min] blocked B/US: 13,194,802 pairs (16.2/other), recall 0.9913
[ 184.7 min]   cross-encoder scored 85,597 ambiguous pairs
[ 184.8 min] B: blocking recall 0.9902, after stage-1 pruning 0.9864, pairs 1,145,479
[ 197.7 min] blocked T/France: 24,166,953 pairs
[ 206.1 min]   cross-encoder scored 462,937 ambiguous pairs
[ 323.5 min] blocked T/India: 79,715,306 pairs
[ 342.3 min]   cross-encoder scored 786,594 ambiguous pairs
[ 421.4 min] blocked T/US: 61,072,310 pairs
[ 437.0 min]   cross-encoder scored 809,101 ambiguous pairs
[ 437.1 min] test: 9,853,007 candidate pairs


## 11. Decision: assignment → per-S1 subset selection, tuned for macro F0.5
1. **Blend** `p = (1-w)·p2 + w·pce` on the ambiguous band.
2. **Assignment:** each S2/S3 record keeps only its best S1, because the data guarantees at most one owner.
3. **Per-S1 subset**, two strategies compared on B:
   * `thr`: keep pairs with p ≥ t.
   * `ef` (expected-F0.5): for S1 candidates sorted by p, with k kept, C = Σtop-k p and G = Σall p, `E[F0.5] ≈ 1.25·C / (0.25·G + k)`. The "predict nothing" option scores `P(no true match) = Π(1-p)`. Take the best k, scale the empty option by λ, and apply a floor `tmin`. This directly optimises the metric, *including* the 1.0 for correct singletons.

The scorer below re-implements the official metric (macro over **all** S1 of the world, singletons included). It is checked against the worked example from the problem statement.

In [16]:
def macro_f05(pred, gt, all_s1):
    tp = pred.merge(gt, on=['s1', 'o']).groupby('s1').size()
    d = pd.DataFrame({'tp': tp, 'np': pred.groupby('s1').size(), 'ng': gt.groupby('s1').size()})
    d = d.reindex(all_s1).fillna(0)
    fp, fn = d.np - d.tp, d.ng - d.tp
    f = np.where((d.np == 0) & (d.ng == 0), 1.0, 1.25 * d.tp / np.maximum(1.25 * d.tp + 0.25 * fn + fp, 1e-9))
    return float(f.mean()), float(d.tp.sum() / max(1, d.np.sum())), float(d.tp.sum() / max(1, d.ng.sum()))

_p = pd.DataFrame({'s1': [1, 1, 1, 2], 'o': [47, 193, 812, 9]})
_g = pd.DataFrame({'s1': [1, 1], 'o': [47, 812]})
assert abs(macro_f05(_p[_p.s1 == 1], _g, [1])[0] - 0.7142857) < 1e-6      # example from the statement
assert macro_f05(_p, _g, [1, 2])[0] == (0.7142857142857143 + 0.0) / 2      # false merge on a singleton = 0
assert macro_f05(_p[_p.s1 == 1], _g, [1, 3])[0] == (0.7142857142857143 + 1.0) / 2   # correct empty = 1

BEST_A = 0.0
def blend(P, w, a=None):
    a = BEST_A if a is None else a
    q = np.where(np.isnan(P.pset.values), P.p2.values, (1 - a) * P.p2.values + a * P.pset.values)
    return np.where(np.isnan(P.pce.values), q, (1 - w) * q + w * P.pce.values)

def assign(P, p):
    Q = P[['s1', 'o']].assign(p=p)
    return Q.sort_values('p', ascending=False, kind='stable').drop_duplicates('o').reset_index(drop=True)

def select(Q, mode, t=0.5, lam=1.0, tmin=0.0):
    if mode == 'thr':
        return Q[Q.p >= t]
    Q = Q.sort_values(['s1', 'p'], ascending=[True, False], kind='stable').reset_index(drop=True)
    g = Q.groupby('s1', sort=False)
    k = g.cumcount().to_numpy() + 1
    C = g.p.cumsum().to_numpy()
    G = g.p.transform('sum').to_numpy()
    ef = 1.25 * C / (0.25 * G + k)
    Q['ef'], Q['k'] = ef, k
    Q['l0'] = np.log1p(-np.clip(Q.p.to_numpy(), 0, 1 - 1e-6))
    g = Q.groupby('s1', sort=False)
    best = g.ef.transform('max').to_numpy()
    kbest = Q.k.where(Q.ef == best).groupby(Q.s1).transform('min').to_numpy()
    p0 = np.exp(g.l0.transform('sum').to_numpy())
    keep = (k <= kbest) & (lam * p0 < best) & (Q.p.to_numpy() >= tmin)
    return Q.loc[keep, ['s1', 'o', 'p']]

allB = WORLD_S1_KEYS['B']
res = []
TGRID = np.arange(0.20, 0.96, 0.05)
for a in ([0.0, 0.25, 0.5, 0.75, 1.0] if SETR is not None else [0.0]):       # 1) set-ranker weight
    Q = assign(PB, blend(PB, 0.0, a))
    for t in TGRID:
        res.append(('thr', a, 0.0, round(t, 2), None, None, *macro_f05(select(Q, 'thr', t=t), GT['B'], allB)))
best_a = max(res, key=lambda r: r[6])[1]
for w in ([0.25, 0.5, 0.75, 1.0] if CE else []):                               # 2) cross-encoder weight
    Q = assign(PB, blend(PB, w, best_a))
    for t in TGRID:
        res.append(('thr', best_a, w, round(t, 2), None, None, *macro_f05(select(Q, 'thr', t=t), GT['B'], allB)))
best_w = max(res, key=lambda r: r[6])[2]
Q = assign(PB, blend(PB, best_w, best_a))
for lam in (0.5, 0.75, 1.0, 1.5, 2.0):                                          # 3) expected-F0.5 rule
    for tmin in (0.0, 0.05, 0.1, 0.2, 0.3):
        res.append(('ef', best_a, best_w, None, lam, tmin, *macro_f05(select(Q, 'ef', lam=lam, tmin=tmin), GT['B'], allB)))
R = pd.DataFrame(res, columns=['mode', 'a', 'w', 't', 'lam', 'tmin', 'F05', 'P', 'R']).sort_values('F05', ascending=False)
print(R.head(15).to_string(index=False))
for a in sorted(R.a.unique()):
    print(f'  best with set-ranker weight a={a}: F0.5 {R[R.a == a].F05.max():.5f}')
BEST = R.iloc[0].to_dict()
BEST_A = float(BEST['a'])
ceiling = macro_f05(PB.loc[PB.y == 1, ['s1', 'o']], GT['B'], allB)[0]
log(f'BEST on B: {BEST} | oracle ceiling given our candidates: {ceiling:.4f}')
_best_sel = select(assign(PB, blend(PB, BEST['w'])), BEST['mode'], t=BEST['t'] if BEST['t'] == BEST['t'] else 0.5,
                   lam=BEST['lam'] if BEST['lam'] == BEST['lam'] else 1.0, tmin=BEST['tmin'] if BEST['tmin'] == BEST['tmin'] else 0.0)
for c in COUNTRIES['B']:
    ks = load_frames('B', c)[0].key.values
    f, p, r = macro_f05(_best_sel[_best_sel.s1.isin(ks)], GT['B'][GT['B'].s1.isin(ks)], ks)
    log(f'  B/{c}: F0.5 {f:.4f}  P {p:.4f}  R {r:.4f}  (S1={len(ks):,})')
json.dump({'best': BEST, 'ceiling': ceiling, 'stats': {k: dict(v) for k, v in STATS.items()}, 'cfg': CFG},
          open(WORK / 'metrics.json', 'w'), indent=1, default=str)

mode   a   w    t  lam  tmin      F05        P        R
 thr 0.5 0.5 0.55  NaN   NaN 0.988581 0.996926 0.972960
 thr 0.5 0.5 0.50  NaN   NaN 0.988528 0.995776 0.975458
 thr 0.5 0.5 0.60  NaN   NaN 0.988258 0.997755 0.970524
 thr 0.5 0.5 0.65  NaN   NaN 0.987673 0.998326 0.968052
 thr 0.5 0.5 0.45  NaN   NaN 0.987623 0.994176 0.977082
  ef 0.5 0.5  NaN  1.0  0.10 0.987583 0.998714 0.963228
  ef 0.5 0.5  NaN  1.0  0.05 0.987583 0.998714 0.963228
  ef 0.5 0.5  NaN  1.0  0.00 0.987583 0.998714 0.963228
  ef 0.5 0.5  NaN  1.0  0.20 0.987583 0.998714 0.963228
  ef 0.5 0.5  NaN  1.0  0.30 0.987583 0.998714 0.963228
  ef 0.5 0.5  NaN  1.5  0.05 0.987504 0.998862 0.963069
  ef 0.5 0.5  NaN  1.5  0.10 0.987504 0.998862 0.963069
  ef 0.5 0.5  NaN  1.5  0.20 0.987504 0.998862 0.963069
  ef 0.5 0.5  NaN  1.5  0.30 0.987504 0.998862 0.963069
  ef 0.5 0.5  NaN  1.5  0.00 0.987504 0.998862 0.963069
  best with set-ranker weight a=0.0: F0.5 0.98146
  best with set-ranker weight a=0.25: F0.5 0.98315
  b

In [17]:
# Error analysis on B: what do our false merges and misses look like?
Qb = select(assign(PB, blend(PB, BEST['w'])), BEST['mode'], t=BEST['t'] or 0.5,
            lam=BEST['lam'] or 1.0, tmin=BEST['tmin'] or 0.0)
m = Qb.merge(GT['B'].assign(hit=1), on=['s1', 'o'], how='left')
fp = m[m.hit.isna()].sample(min(10, int(m.hit.isna().sum())), random_state=0)
fn = GT['B'].merge(Qb, on=['s1', 'o'], how='left')
fn = fn[fn.p.isna()].sample(min(10, int(fn.p.isna().sum())), random_state=0)
look = pd.concat([pd.concat(load_frames('B', c)) for c in COUNTRIES['B']]).set_index('key')
def show(df, title):
    print(f'--- {title}')
    for s, o in zip(df.s1, df.o):
        print(f"{look.at[s, 'business_name']!r:45} {look.at[s, 'business_address'][:50]!r}\n"
              f"   ~ {look.at[o, 'business_name']!r:41} {look.at[o, 'business_address'][:50]!r}")
show(fp, 'false merges (hurt precision)')
show(fn, 'missed matches (hurt recall)')
del look

--- false merges (hurt precision)
'Szatkowski Crystal First PC'                 '3530 Val Vista Drive, Unit 202, Gilbert, AZ'
   ~ 'Szatkowski Szatkowski Crystal Fifth'     '3530 Val Vista Drive, # 202, Gilbert, Arizona'
'Grams, Tabbi, DMD, P.C. PC'                  '9112 Comanche Road, Albuquerque, NM'
   ~ 'Grams, Teabi, DMD, [P.C.]'               ''
'Kolkata Nutrients Private Limited'           'C/O Rampratap Somani, 14 Noormal Lohia Lane, Kolka'
   ~ 'Kolkata Nutrients Limited Center'        ''
'Surgical Care Center'                        '1301 Fordham Boulevard, Unit 231M, Chapel Hill, NC'
   ~ 'Arcxylo'                                 '1301 FORDHAM BOULEVARD, PMB 6823, CHAPELL HILL, NC'
'Brite Preparatory Academy'                   '1153 Oak Wood Lane, Washington City, UT'
   ~ 'Brite Preparatory Academy LP'            ''
'Star Seva Samiti'                            'E608 Prahladnagar Trade, B/H Titanium Trade Center'
   ~ 'SE'                                      'Ahmadabad Ci

## 12. Write the submission files and validate
* `matching_results.tsv`: one row for **every** test S1 (including France and ones with no candidates), empty list = singleton.
* `candidate_pairs.tsv`: the stage-1-pruned set that stage 2 scored. Matches are a subset of it by construction.

Then the official validator runs.

In [18]:
def write_lists(pairs, path, col):
    ids = pd.DataFrame({'s1': skey(pairs.s1.values).values, 'o': skey(pairs.o.values).values})
    lists = ids.groupby('s1').o.agg(','.join)
    out = pd.DataFrame({'source1_entity_id': TEST_S1_IDS})
    out[col] = out.source1_entity_id.map(lists).fillna('')
    out.to_csv(path, sep='\t', index=False, quoting=csv.QUOTE_NONE)
    log(f'wrote {path.name}: {len(out):,} rows, {int((out[col] != "").sum()):,} non-empty')

final = select(assign(PT, blend(PT, BEST['w'])), BEST['mode'], t=BEST['t'] or 0.5,
               lam=BEST['lam'] or 1.0, tmin=BEST['tmin'] or 0.0)
write_lists(PT[['s1', 'o']].drop_duplicates(), OUT / 'candidate_pairs.tsv', 'candidate_entity_ids')
write_lists(final, OUT / 'matching_results.tsv', 'matched_entity_ids')
print(pd.read_csv(OUT / 'matching_results.tsv', sep='\t', dtype=str, keep_default_na=False).head(10).to_string())

[ 439.9 min] wrote candidate_pairs.tsv: 1,732,544 rows, 1,728,385 non-empty
[ 440.5 min] wrote matching_results.tsv: 1,732,544 rows, 1,633,754 non-empty
  source1_entity_id                                                            matched_entity_ids
0      S1-714132312                                        S3-625880872,S2-637340732,S2-187020300
1      S1-106407869                                        S3-613056593,S3-585937637,S2-705547832
2      S1-156285671              S2-172049742,S3-871416040,S2-648730048,S3-788875930,S3-339493107
3      S1-689823050               S2-65478038,S2-212601842,S2-319486263,S2-829077756,S3-506456729
4      S1-921369899  S3-285097097,S2-995426671,S2-107533354,S3-779830725,S2-808920974,S3-57594330
5      S1-481669221                                                                  S2-284602321
6      S1-909865979                                                     S3-699465428,S2-528901108
7      S1-280204013                                            

In [19]:
try:
    VALIDATOR = _find('validate_submission.py')
    !python "{VALIDATOR}" --matching "{OUT}/matching_results.tsv" --candidate "{OUT}/candidate_pairs.tsv" --test-dir "{TEST_DIR}"
except FileNotFoundError:
    print('validate_submission.py not in the inputs - add utils/ to the dataset to run the official check')
    m = pd.read_csv(OUT / 'matching_results.tsv', sep='\t', dtype=str, keep_default_na=False)
    assert m.source1_entity_id.is_unique and set(m.source1_entity_id) == set(TEST_S1_IDS)
    assert m.matched_entity_ids.map(lambda s: len(s.split(',')) == len(set(s.split(','))) if s else True).all()
    print('basic checks PASS')

ML Challenge 2026 — submission validator
  test dir: /kaggle/input/datasets/saumya170408/amazon-ml-challenge-26/student_resource/dataset/test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (98790 empty, 1633754 non-empty).
  candidate_pairs.tsv: 1732544 rows (4159 empty, 1728385 non-empty).

PASS — no blocking issues found. Safe to submit.


## 13. Pin the environment (for the submission package)
`requirements.txt` is written with the exact versions this run used, so our package can be reproduced.

In [20]:
import importlib.metadata as md_
pkgs = ['numpy', 'pandas', 'pyarrow', 'scipy', 'scikit-learn', 'xgboost', 'torch', 'transformers',
        'rapidfuzz', 'sparse_dot_topn', 'unidecode']
def _ver(p):
    try:
        return f'{p}=={md_.version(p)}'
    except md_.PackageNotFoundError:
        return f'# {p} (version not found)'
req = '\n'.join(map(_ver, pkgs)) + '\n'
(WORK / 'requirements.txt').write_text(req)
print(req)
log('done. Download /kaggle/working/output/*.tsv, metrics.json and requirements.txt')

numpy==2.0.2
pandas==2.3.3
pyarrow==24.0.0
scipy==1.16.3
scikit-learn==1.6.1
xgboost==3.2.0
torch==2.10.0+cu128
transformers==5.0.0
rapidfuzz==3.14.6
sparse_dot_topn==1.2.0
unidecode==1.4.0

[ 441.2 min] done. Download /kaggle/working/output/*.tsv, metrics.json and requirements.txt
